In [16]:
# 1 library setup

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import os
import glob
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score


In [17]:
# column def info

ALL_COLUMNS = [
    'HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2',
    'BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2','AST','BUN',
    'Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct',
    'Glucose','Lactate','Magnesium','Phosphate','Potassium',
    'Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen',
    'Platelets','Age','Gender','Unit1','Unit2','HospAdmTime','ICULOS',
    'SepsisLabel'
]

# Demographics (static or quasi-static)
DEMO_COLS = ['Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime']

# Dynamic clinical variables (everything else except label)
LABEL_COL = 'SepsisLabel'

DYNAMIC_COLS = [c for c in ALL_COLUMNS if c not in DEMO_COLS and c != LABEL_COL]

In [18]:
# multi horizon label construction

def make_multihorizon_labels(sepsis_label, horizons=[1,3,6,12]):
    """
    sepsis_label: [T] array, 0/1, with a single transition to 1 at onset
    Returns: Y [T, 4]
    """
    T = len(sepsis_label)
    Y = np.zeros((T, len(horizons)), dtype=np.float32)

    if sepsis_label.max() == 0:
        return Y

    ts = np.where(sepsis_label == 1)[0][0]  # onset time index

    for i, d in enumerate(horizons):
        for t in range(T):
            if t <= ts < t + d:
                Y[t, i] = 1.0

    return Y


In [19]:
# load single patient file

def load_patient_file(path):
    df = pd.read_csv(path, sep='|')

    # Extract dynamic features
    X = df[DYNAMIC_COLS].values.astype(np.float32)

    # Replace missing with NaN (already usually is)
    X[X == -1] = np.nan

    # Demographics: take first non-NaN
    demo = []
    for c in DEMO_COLS:
        col = df[c].values
        idx = np.where(~pd.isna(col))[0]
        if len(idx) == 0:
            demo.append(0.0)
        else:
            demo.append(col[idx[0]])
    demo = np.array(demo, dtype=np.float32)
    demo = np.nan_to_num(demo, nan=0.0, posinf=0.0, neginf=0.0)

    # Sepsis labels
    sepsis = df[LABEL_COL].values.astype(np.int32)

    Y = make_multihorizon_labels(sepsis)

    return X, demo, Y


In [20]:
# load whole dataset

def load_physionet_dataset(root_dir):
    files = sorted(glob.glob(os.path.join(root_dir, "*.psv")))

    patient_X = []
    patient_demo = []
    patient_Y = []

    for f in tqdm(files):
        try:
            X, demo, Y = load_patient_file(f)
            patient_X.append(X)
            patient_demo.append(demo)
            patient_Y.append(Y)
        except Exception as e:
            print("Failed on", f, e)

    return patient_X, patient_demo, patient_Y


In [21]:
# data split on patient level

def split_dataset(patient_X, patient_demo, patient_Y, seed=42):
    N = len(patient_X)
    rng = np.random.RandomState(seed)
    idx = np.arange(N)
    rng.shuffle(idx)

    n_train = int(0.7 * N)
    n_val = int(0.15 * N)

    train_idx = idx[:n_train]
    val_idx = idx[n_train:n_train+n_val]
    test_idx = idx[n_train+n_val:]

    def subset(idxs):
        return (
            [patient_X[i] for i in idxs],
            [patient_demo[i] for i in idxs],
            [patient_Y[i] for i in idxs],
        )

    return subset(train_idx), subset(val_idx), subset(test_idx)


In [22]:
# 2 preprocessing unit

class PhysioNetPreprocessor:
    def __init__(self, pop_mean, pop_std):
        self.pop_mean = pop_mean
        self.pop_std = pop_std

    def transform(self, X, demo=None):
        """
        X: [T, F] with NaNs
        demo: [D] or None
        Returns: [T, 3F + D]
        """
        T, F = X.shape

        mask = (~np.isnan(X)).astype(np.float32)

        delta = np.zeros_like(X, dtype=np.float32)
        last_seen = np.zeros(F)

        for t in range(T):
            for f in range(F):
                if mask[t, f] == 1:
                    delta[t, f] = 0
                    last_seen[f] = t
                else:
                    delta[t, f] = t - last_seen[f]

        delta = np.clip(delta, 0, 72) / 24.0

        # Forward fill
        X_filled = X.copy()
        for f in range(F):
            col = X_filled[:, f]
            idx = np.where(~np.isnan(col))[0]
            if len(idx) == 0:
                X_filled[:, f] = self.pop_mean[f]
            else:
                last = col[idx[0]]
                for t in range(T):
                    if not np.isnan(col[t]):
                        last = col[t]
                    else:
                        col[t] = last

        # Any remaining NaNs? (safety)
        X_filled = np.nan_to_num(X_filled, nan=self.pop_mean)

        X_norm = (X_filled - self.pop_mean) / (self.pop_std + 1e-6)

        parts = [X_norm, mask, delta]
        if demo is not None:
            demo_rep = np.repeat(demo[None, :], T, axis=0)
            parts.append(demo_rep)

        U = np.concatenate(parts, axis=1)
        return torch.tensor(U, dtype=torch.float32)


In [23]:
# 3 dataset wrapper

class PhysioNetDataset(Dataset):
    def __init__(self, patient_X, patient_demo, labels, preprocessor):
        self.patient_X = patient_X
        self.patient_demo = patient_demo
        self.labels = labels  # [T, 4]
        self.prep = preprocessor

    def __len__(self):
        return len(self.patient_X)

    def __getitem__(self, idx):
        X = self.patient_X[idx]
        demo = self.patient_demo[idx]
        Y = self.labels[idx]
        U = self.prep.transform(X, demo)
        return U, torch.tensor(Y, dtype=torch.float32)

In [24]:
# 4 variable length batching

def collate_fn(batch):
    Us, Ys = zip(*batch)
    lengths = [u.shape[0] for u in Us]
    Tm = max(lengths)
    B = len(Us)
    D = Us[0].shape[1]

    U_pad = torch.zeros(B, Tm, D)
    Y_pad = torch.zeros(B, Tm, 4)
    M_pad = torch.zeros(B, Tm)

    for i, (u, y) in enumerate(zip(Us, Ys)):
        T = u.shape[0]
        U_pad[i, :T] = u
        Y_pad[i, :T] = y
        M_pad[i, :T] = 1

    U_pad = torch.nan_to_num(U_pad, nan=0.0, posinf=0.0, neginf=0.0)
    return U_pad, Y_pad, M_pad, torch.tensor(lengths)


In [25]:
# 5 LNN model

class MonolithicLNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, tau=4.0, substeps=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.tau = tau
        self.substeps = substeps

        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.V = nn.Linear(input_dim, hidden_dim, bias=True)

        self.readout = nn.Linear(hidden_dim, 4)  # 4 horizons

    def step(self, x, u, dt):
        """
        One Euler step
        """
        dx = -x.double() + torch.tanh(self.W(x) + self.V(u)).double()
        return x + (dt / self.tau) * dx

    def forward(self, U, pad_mask):
        """
        U: [B, T, D]
        pad_mask: [B, T]
        """
        if not torch.isfinite(U).all():
            print("NaNs in input!")
            exit()
        B, T, D = U.shape
        device = U.device

        x = torch.zeros(B, self.hidden_dim, device=device)
        x = x.double()
        outputs = []

        dt = 1.0 / self.substeps

        for t in range(T):
            u_t = U[:, t]
            u_t = u_t.double()
            # sub-stepping solver
            for _ in range(self.substeps):
                x = self.step(x, u_t, dt)

            logits = self.readout(x)
            outputs.append(logits)

        outputs = torch.stack(outputs, dim=1)  # [B, T, 4]
        return outputs


In [32]:
#reporting for AUROC and AUPRC

@torch.no_grad()
def compute_metrics(model, loader, device):
    model.eval()

    # For each horizon, we accumulate a flat list
    all_probs = {0: [], 1: [], 2: [], 3: []}
    all_targets = {0: [], 1: [], 2: [], 3: []}

    for U, Y, M, L in loader:
        U = U.to(device)
        M = M.to(device)

        logits = model(U, M)
        probs = torch.sigmoid(logits)  # [B, T, 4]

        B, T, _ = probs.shape

        for k in range(4):
            probs_k = probs[:, :, k]      # [B, T]
            targets_k = Y[:, :, k].to(device)  # [B, T]

            # Use mask to remove padding
            mask = M > 0                  # [B, T]

            probs_k = probs_k[mask]       # [N]
            targets_k = targets_k[mask]   # [N]

            all_probs[k].append(probs_k.cpu())
            all_targets[k].append(targets_k.cpu())

    metrics = {}

    for k, horizon in enumerate([1, 3, 6, 12]):
        probs_k = torch.cat(all_probs[k], dim=0).numpy()
        targets_k = torch.cat(all_targets[k], dim=0).numpy()

        # Edge case: if only one class present
        if len(np.unique(targets_k)) < 2:
            auroc = np.nan
            auprc = np.nan
        else:
            auroc = roc_auc_score(targets_k, probs_k)
            auprc = average_precision_score(targets_k, probs_k)

        metrics[horizon] = {
            "AUROC": float(auroc),
            "AUPRC": float(auprc)
        }

    # Macro average
    aurocs = [metrics[h]["AUROC"] for h in [1,3,6,12] if not np.isnan(metrics[h]["AUROC"])]
    auprcs = [metrics[h]["AUPRC"] for h in [1,3,6,12] if not np.isnan(metrics[h]["AUPRC"])]

    metrics["macro"] = {
        "AUROC": float(np.mean(aurocs)) if len(aurocs) else np.nan,
        "AUPRC": float(np.mean(auprcs)) if len(auprcs) else np.nan,
    }

    return metrics

def print_metrics(metrics, prefix=""):
    line = prefix
    for h in [1,3,6,12]:
        m = metrics[h]
        line += f" | {h}h AUROC {m['AUROC']:.3f} AUPRC {m['AUPRC']:.3f}"
    line += f" | Macro AUPRC {metrics['macro']['AUPRC']:.3f}"
    print(line)

In [27]:
# 6 multi horizon loss function

def multi_horizon_loss(logits, targets, mask, pos_weights):
    """
    logits: [B, T, 4]
    targets: [B, T, 4]
    mask: [B, T]
    pos_weights: [4]
    """
    loss = 0
    for k in range(4):
        bce = F.binary_cross_entropy_with_logits(
            logits[:, :, k],
            targets[:, :, k],
            reduction="none",
            pos_weight=pos_weights[k]
        )
        bce = bce * mask
        loss += bce.sum() / mask.sum()

    return loss


In [28]:
# 7 training loop

def train_epoch(model, loader, optimizer, device, pos_weights):
    model.double()
    model.train()
    total = 0

    for U, Y, M, L in tqdm(loader):
        U = U.to(device)
        Y = Y.to(device)
        M = M.to(device)

        U = U.double()
        Y = Y.double()
        M = M.double()

        optimizer.zero_grad()
        logits = model(U, M)
        loss = multi_horizon_loss(logits, Y, M, pos_weights)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total += loss.item()

    return total / len(loader)


In [29]:
# 8 validation

@torch.no_grad()
def eval_epoch(model, loader, device):
    model.double()
    model.eval()
    all_logits = []
    all_targets = []
    all_masks = []

    for U, Y, M, L in loader:
        U = U.to(device)
        U = U.double()
        Y = Y.double()
        M = M.double()
        logits = model(U, M)

        all_logits.append(torch.sigmoid(logits).cpu())
        all_targets.append(Y)
        all_masks.append(M)

    return torch.cat(all_logits), torch.cat(all_targets), torch.cat(all_masks)


In [30]:
def check_dataset_for_nans(train_X, train_demo, train_Y, name="train"):
    print(f"Checking {name} dataset...")

    bad = False

    # Check X
    for i, X in enumerate(train_X):
        if not np.isfinite(X).all():
            idx = np.where(~np.isfinite(X))
            print(f"[{name}] NaN/Inf in train_X[{i}] at positions {list(zip(idx[0][:5], idx[1][:5]))} ...")
            bad = True
            break

    # Check demo
    for i, d in enumerate(train_demo):
        if not np.isfinite(d).all():
            idx = np.where(~np.isfinite(d))
            print(f"[{name}] NaN/Inf in train_demo[{i}] at positions {idx[0][:5]} ...")
            bad = True
            break

    # Check Y
    for i, Y in enumerate(train_Y):
        if not np.isfinite(Y).all():
            idx = np.where(~np.isfinite(Y))
            print(f"[{name}] NaN/Inf in train_Y[{i}] at positions {list(zip(idx[0][:5], idx[1][:5]))} ...")
            bad = True
            break

    if not bad:
        print(f"[{name}] No NaNs or Infs found in dataset.")
    else:
        print(f"[{name}] Dataset contains invalid values.")


In [33]:
# 9 code run


DATA_DIR = "data/physionet/training_setA/training"

patient_X, patient_demo, patient_Y = load_physionet_dataset(DATA_DIR)

(train_X, train_demo, train_Y), (val_X, val_demo, val_Y), (test_X, test_demo, test_Y) = \
    split_dataset(patient_X, patient_demo, patient_Y)

check_dataset_for_nans(train_X, train_demo, train_Y, name="train")
check_dataset_for_nans(val_X, val_demo, val_Y, name="val")
check_dataset_for_nans(test_X, test_demo, test_Y, name="test")

print("Train patients:", len(train_X))
print("Val patients:", len(val_X))
print("Test patients:", len(test_X))

#num_features = train_X[0].shape[1]
#demo_dim = train_demo[0].shape[0]

pop_mean = np.nanmean(np.concatenate(train_X, axis=0), axis=0)
pop_std  = np.nanstd(np.concatenate(train_X, axis=0), axis=0)

pop_mean = np.nan_to_num(pop_mean, nan=0.0)
pop_std = np.nan_to_num(pop_std, nan=1.0)
pop_std[pop_std < 1e-6] = 1.0

prep = PhysioNetPreprocessor(pop_mean, pop_std)

ds_train = PhysioNetDataset(train_X, train_demo, train_Y, prep)
ds_val   = PhysioNetDataset(val_X, val_demo, val_Y, prep)

dl_train = DataLoader(ds_train, batch_size=32, shuffle=True, collate_fn=collate_fn)
dl_val   = DataLoader(ds_val, batch_size=32, shuffle=False, collate_fn=collate_fn)

device = "cuda"

model = MonolithicLNN(
    input_dim=ds_train[0][0].shape[1],
    hidden_dim=256,
    tau=4.0,
    substeps=4
).to(device)
model.double()

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

pos_weights = torch.tensor([10.0, 5.0, 3.0, 2.0]).to(device)

for epoch in range(20):
    train_loss = train_epoch(model, dl_train, optimizer, device, pos_weights)    
    print("Epoch", epoch, "train loss", train_loss)

    val_metrics = compute_metrics(model, dl_val, device)
    print_metrics(val_metrics, prefix="Val")


100%|██████████| 20336/20336 [00:49<00:00, 409.74it/s]
C:\Users\david\AppData\Local\Temp\ipykernel_21868\1913505607.py:22: RuntimeWarning: Mean of empty slice
  pop_mean = np.nanmean(np.concatenate(train_X, axis=0), axis=0)


Checking train dataset...
[train] NaN/Inf in train_X[0] at positions [(0, 7), (0, 10), (0, 12), (0, 13), (0, 14)] ...
[train] Dataset contains invalid values.
Checking val dataset...
[val] NaN/Inf in train_X[0] at positions [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)] ...
[val] Dataset contains invalid values.
Checking test dataset...
[test] NaN/Inf in train_X[0] at positions [(0, 7), (0, 13), (0, 17), (0, 19), (0, 20)] ...
[test] Dataset contains invalid values.
Train patients: 14235
Val patients: 3050
Test patients: 3051


C:\Users\david\AppData\Roaming\Python\Python310\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
100%|██████████| 445/445 [03:10<00:00,  2.33it/s]


Epoch 0 train loss 0.640056235591487
Val | 1h AUROC 0.711 AUPRC 0.009 | 3h AUROC 0.714 AUPRC 0.017 | 6h AUROC 0.715 AUPRC 0.032 | 12h AUROC 0.712 AUPRC 0.062 | Macro AUPRC 0.030


100%|██████████| 445/445 [03:18<00:00,  2.25it/s]


...

Epoch 17 train loss 0.47853430189555907
Val | 1h AUROC 0.791 AUPRC 0.010 | 3h AUROC 0.784 AUPRC 0.026 | 6h AUROC 0.785 AUPRC 0.048 | 12h AUROC 0.788 AUPRC 0.092 | Macro AUPRC 0.044


100%|██████████| 445/445 [03:07<00:00,  2.37it/s]


Epoch 18 train loss 0.4733880216955518
Val | 1h AUROC 0.786 AUPRC 0.010 | 3h AUROC 0.777 AUPRC 0.026 | 6h AUROC 0.777 AUPRC 0.047 | 12h AUROC 0.776 AUPRC 0.087 | Macro AUPRC 0.043


100%|██████████| 445/445 [03:06<00:00,  2.38it/s]


Epoch 19 train loss 0.47127104331144437
Val | 1h AUROC 0.798 AUPRC 0.012 | 3h AUROC 0.794 AUPRC 0.028 | 6h AUROC 0.794 AUPRC 0.051 | 12h AUROC 0.796 AUPRC 0.095 | Macro AUPRC 0.047
